# Proyek Analisis Data: Bike Sharing Dataset
- Nama: Daffa Kumara Khiar Faisa
- Email: daffakurama10@gmail.com
- Id Dicoding: 

> **Catatan revisi (v2):** Pertanyaan 1 & 2 versi awal (musim terbanyak & 10 hari terakhir) diganti dengan pertanyaan yang lebih berbobot menggunakan teknik advanced: **time series decomposition** dan **clustering (K-Means)**. Pertanyaan 3 & 4 (pengaruh cuaca, pola hari kerja vs libur) dipertahankan dari revisi sebelumnya.

## Menentukan Pertanyaan Bisnis

- Pertanyaan 1 = Bagaimana tren pertumbuhan penyewaan sepeda dari tahun 2011 ke 2012, dan pola musiman apa yang terlihat dari dekomposisi time series?
- Pertanyaan 2 = Bisakah hari-hari dikelompokkan ke dalam beberapa segmen berdasarkan karakteristik cuaca & penyewaan, dan apa profil dari masing-masing segmen?
- Pertanyaan 3 = Bagaimana pengaruh kondisi cuaca (suhu, kelembapan, kecepatan angin, kondisi langit) terhadap jumlah penyewaan sepeda?
- Pertanyaan 4 = Bagaimana perbedaan pola penyewaan casual vs registered antara hari kerja dan hari libur/akhir pekan?

## Menyiapkan semua library yang dibutuhkan

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

## Data Wrangling

### Gathering Data

In [ ]:
day_df = pd.read_csv("./Data/day.csv")
day_df.head()

### Assessing Data

In [ ]:
# Menilai data day_df
day_df.info()

In [ ]:
# Mengidentifikasi missing value pada day_df
day_df.isnull().sum()

In [ ]:
# Memeriksa duplikasi data day_df
print("Jumlah Duplikasi : ", day_df.duplicated().sum())

In [ ]:
# Menampilkan ringkasan parameter statistik data day_df
day_df.describe()

### Cleaning Data

In [ ]:
# Menghilangkan Duplicate Data
day_df.drop_duplicates(inplace=True)
print("Jumlah Duplikasi : ", day_df.duplicated().sum())

In [ ]:
# Menangani Missing Value pada data day_df
day_df.isna().sum()

In [ ]:
# Konversi dteday ke datetime & mapping label season untuk mempermudah pembacaan chart
day_df['dteday'] = pd.to_datetime(day_df['dteday'])

season_map = {1: 'Springer', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
day_df['season_label'] = day_df['season'].map(season_map)

## Exploratory Data Analysis (EDA)

### Explore ...

In [ ]:
# Mengexplorasi data day_df
day_df.describe(include="all")

In [ ]:
# Mengeksplor banyaknya jumlah data casual pada day_df
print("Jumlah penyewa casual adalah : ", day_df.casual.sum())

In [ ]:
# Mengeksplor banyaknya jumlah data registered pada day_df
print("Jumlah penyewa registered adalah : ", day_df.registered.sum())

## Visualization & Explanatory Analysis

### Pertanyaan 1: Bagaimana tren pertumbuhan penyewaan sepeda dari tahun 2011 ke 2012, dan pola musiman apa yang terlihat dari dekomposisi time series?

In [ ]:
# Agregasi total penyewaan per bulan, lalu bandingkan pertumbuhan 2011 vs 2012
monthly_df = day_df.set_index('dteday').resample('MS')['cnt'].sum()

total_2011 = day_df[day_df['yr'] == 0]['cnt'].sum()
total_2012 = day_df[day_df['yr'] == 1]['cnt'].sum()
growth_pct = (total_2012 - total_2011) / total_2011 * 100

print(f"Total penyewaan 2011 : {total_2011:,}")
print(f"Total penyewaan 2012 : {total_2012:,}")
print(f"Pertumbuhan YoY      : {growth_pct:.1f}%")

In [ ]:
# Dekomposisi time series (trend, seasonal, residual) dari data bulanan
result = seasonal_decompose(monthly_df, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True)
result.observed.plot(ax=axes[0], color='#72BCD4')
axes[0].set_title('Observed (Total Penyewaan Bulanan)')
result.trend.plot(ax=axes[1], color='#1f6f8b')
axes[1].set_title('Trend')
result.seasonal.plot(ax=axes[2], color='#e07b39')
axes[2].set_title('Seasonal')
result.resid.plot(ax=axes[3], color='gray', marker='o', linestyle='None')
axes[3].set_title('Residual')
plt.suptitle('Dekomposisi Time Series: Total Penyewaan Sepeda Bulanan (2011-2012)', fontsize=13, y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi tren pertumbuhan bulanan beserta garis trend
plt.figure(figsize=(11, 5.5))
plt.plot(monthly_df.index, monthly_df.values, marker='o', color='#72BCD4', label='Total Penyewaan Bulanan')
plt.plot(monthly_df.index, result.trend, color='#1f6f8b', linewidth=2, label='Trend (moving average)')
plt.title('Tren Pertumbuhan Penyewaan Sepeda Bulanan (2011 vs 2012)', fontsize=13)
plt.ylabel('Total Penyewaan (cnt)')
plt.xlabel('Bulan')
plt.legend()
plt.tight_layout()
plt.show()

**Insight:** Penyewaan sepeda tumbuh **64,9%** dari tahun 2011 (1.243.103) ke 2012 (2.049.576) — pertumbuhan bisnis yang sangat kuat secara tahunan. Dari dekomposisi time series terlihat dua elemen jelas: (1) **komponen trend** naik konsisten sepanjang periode, menandakan pertumbuhan bisnis yang riil, bukan sekadar fluktuasi musiman; (2) **komponen seasonal** menunjukkan pola berulang tiap tahun — penyewaan naik dari musim semi, memuncak di pertengahan-akhir tahun (musim panas/gugur), lalu turun tajam di musim dingin (Desember-Januari).

*Catatan metodologis:* karena dataset hanya mencakup 2 siklus tahunan penuh (24 titik bulanan), estimasi komponen seasonal & residual pada dekomposisi ini bersifat indikatif, bukan estimasi yang robust secara statistik. Idealnya, dekomposisi time series butuh data historis lebih panjang (3+ tahun) untuk hasil yang lebih andal.

### Pertanyaan 2: Bisakah hari-hari dikelompokkan ke dalam beberapa segmen berdasarkan karakteristik cuaca & penyewaan, dan apa profil dari masing-masing segmen?

In [ ]:
# Standarisasi fitur numerik yang relevan untuk clustering
cluster_features = ['cnt', 'temp', 'hum', 'windspeed']
X_scaled = StandardScaler().fit_transform(day_df[cluster_features])

# Elbow method untuk menentukan jumlah cluster optimal
inertias = []
K_range = range(1, 8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 5))
plt.plot(list(K_range), inertias, marker='o', color='#72BCD4')
plt.title('Elbow Method untuk Menentukan Jumlah Cluster Optimal', fontsize=13)
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

In [ ]:
# Fit K-Means dengan k=3 (dipilih dari elbow method di atas)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
day_df['cluster'] = kmeans.fit_predict(X_scaled)

# Beri label deskriptif berdasarkan rata-rata cnt tiap cluster
cluster_order = day_df.groupby('cluster')['cnt'].mean().sort_values().index.tolist()
label_map = {cluster_order[0]: 'Sepi', cluster_order[1]: 'Sedang', cluster_order[2]: 'Ramai'}
day_df['cluster_label'] = day_df['cluster'].map(label_map)

profile = day_df.groupby('cluster_label')[cluster_features].mean().reindex(['Ramai', 'Sedang', 'Sepi']).round(3)
profile['jumlah_hari'] = day_df['cluster_label'].value_counts().reindex(['Ramai', 'Sedang', 'Sepi'])
profile

In [ ]:
# Visualisasi scatter: suhu vs jumlah penyewaan, diwarnai berdasarkan segmen
palette = {'Ramai': '#72BCD4', 'Sedang': '#D3D3D3', 'Sepi': '#a3a3a3'}

plt.figure(figsize=(8, 6))
sns.scatterplot(data=day_df, x='temp', y='cnt', hue='cluster_label',
                 hue_order=['Ramai', 'Sedang', 'Sepi'], palette=palette, alpha=0.8, s=50)
plt.title('Segmentasi Hari Berdasarkan Suhu & Jumlah Penyewaan (K-Means, k=3)', fontsize=12)
plt.xlabel('Suhu (normalisasi)')
plt.ylabel('Jumlah Penyewaan (cnt)')
plt.legend(title='Segmen')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi profil rata-rata tiap segmen per fitur
prof = day_df.groupby('cluster_label')[cluster_features].mean().reindex(['Ramai', 'Sedang', 'Sepi'])
colors_bar = ['#72BCD4', '#D3D3D3', '#a3a3a3']

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
for ax, feat in zip(axes, cluster_features):
    ax.bar(prof.index, prof[feat], color=colors_bar)
    ax.set_title(feat)
    for i, v in enumerate(prof[feat]):
        label = f"{v:,.0f}" if feat == 'cnt' else f"{v:.2f}"
        ax.text(i, v, label, ha='center', va='bottom', fontsize=9)
plt.suptitle('Profil Rata-rata Tiap Segmen Hari', fontsize=13)
plt.tight_layout()
plt.show()

**Insight:** Dengan K-Means (k=3, dipilih dari elbow method), hari-hari terbagi menjadi 3 segmen dengan karakteristik jelas:
- **Ramai** (339 hari): suhu hangat (0,65), kelembapan sedang, angin rendah → rata-rata **5.982** penyewaan/hari — kombinasi cuaca ideal untuk bersepeda.
- **Sedang** (196 hari): suhu dingin (0,32), kelembapan rendah, tapi angin paling kencang (0,25) → rata-rata 3.341 penyewaan/hari.
- **Sepi** (196 hari): suhu agak dingin (0,40), **kelembapan tertinggi** (0,77) → rata-rata 3.113 penyewaan/hari — segmen paling sepi.

Segmentasi ini menegaskan bahwa kombinasi **suhu hangat + kelembapan rendah/sedang + angin rendah** adalah kondisi paling optimal untuk mendorong permintaan penyewaan sepeda, dan bisa dipakai sebagai dasar untuk perencanaan operasional (mis. alokasi armada, staffing, atau promosi dinamis berdasarkan prediksi cuaca harian).

### Pertanyaan 3: Bagaimana pengaruh kondisi cuaca terhadap jumlah penyewaan sepeda?

In [ ]:
# Korelasi variabel cuaca (temp, atemp, hum, windspeed) terhadap cnt
corr_matrix = day_df[['temp', 'atemp', 'hum', 'windspeed', 'cnt']].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Korelasi Faktor Cuaca terhadap Jumlah Penyewaan", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Rata-rata penyewaan berdasarkan kondisi cuaca (weathersit)
weather_map = {1: 'Cerah', 2: 'Berkabut/\nBerawan', 3: 'Hujan/Salju\nRingan'}
day_df['weathersit_label'] = day_df['weathersit'].map(weather_map)
order = ['Cerah', 'Berkabut/\nBerawan', 'Hujan/Salju\nRingan']
means = day_df.groupby('weathersit_label')['cnt'].mean().reindex(order)

colors = ["#72BCD4", "#D3D3D3", "#D3D3D3"]
plt.figure(figsize=(7, 5))
ax = sns.barplot(x=order, y=means.values, hue=order, palette=colors, legend=False)
for i, v in enumerate(means.values):
    ax.text(i, v + 50, f"{v:,.0f}", ha='center', fontsize=10)
plt.title("Rata-rata Penyewaan Sepeda per Kondisi Cuaca", fontsize=13)
plt.ylabel("Rata-rata Penyewaan (cnt)")
plt.xlabel("Kondisi Cuaca")
plt.tight_layout()
plt.show()

**Insight:** Suhu (`temp`/`atemp`) memiliki korelasi positif sedang (~0,63) terhadap jumlah penyewaan — semakin hangat cuaca, semakin banyak orang menyewa sepeda. Sebaliknya, `windspeed` (korelasi -0,23) dan `hum` (korelasi -0,10) berhubungan negatif, meski pengaruhnya lebih lemah. Dari sisi kondisi langit, cuaca **cerah** menghasilkan rata-rata 4.877 penyewaan/hari, turun ke 4.036 saat berkabut/berawan, dan anjlok ke 1.803 saat hujan/salju ringan — penurunan sekitar **63%** dibanding cuaca cerah. Ini menunjukkan cuaca adalah salah satu faktor operasional paling berpengaruh terhadap permintaan.

### Pertanyaan 4: Bagaimana perbedaan pola penyewaan casual vs registered antara hari kerja dan hari libur/akhir pekan?

In [ ]:
# Rata-rata casual & registered: hari kerja vs libur/akhir pekan
day_df['tipe_hari'] = day_df['workingday'].map({1: 'Hari Kerja', 0: 'Libur/Akhir Pekan'})
by_daytype = day_df.groupby('tipe_hari')[['casual', 'registered']].mean().reset_index()
melt_df = by_daytype.melt(id_vars='tipe_hari', value_vars=['casual', 'registered'],
                           var_name='tipe_penyewa', value_name='rata2')

plt.figure(figsize=(8, 5.5))
ax = sns.barplot(data=melt_df, x='tipe_hari', y='rata2', hue='tipe_penyewa', palette=['#72BCD4', '#D3D3D3'])
for c in ax.containers:
    ax.bar_label(c, fmt='%.0f')
plt.title("Rata-rata Penyewaan Casual vs Registered:\nHari Kerja vs Libur/Akhir Pekan", fontsize=13)
plt.xlabel("Tipe Hari")
plt.ylabel("Rata-rata Jumlah Penyewa")
plt.legend(title=None)
plt.tight_layout()
plt.show()

**Insight:** Pada **hari kerja**, penyewa registered sangat mendominasi (rata-rata 3.978/hari) dibanding casual (607/hari) — pola ini konsisten dengan penggunaan sepeda untuk **commuting** (berangkat kerja/kuliah). Sebaliknya, pada **libur/akhir pekan**, jumlah casual melonjak lebih dari 2x lipat menjadi 1.371/hari, sementara registered justru sedikit turun ke 2.959/hari — mengindikasikan penggunaan yang lebih condong ke **rekreasi/wisata** saat akhir pekan. Insight ini bisa dipakai untuk strategi bisnis: promosi member baru lebih efektif menyasar pengguna casual di akhir pekan, sementara pada hari kerja fokus retensi member existing.

## Conclusion

- **Kesimpulan pertanyaan 1:** Penyewaan sepeda tumbuh **64,9%** dari 2011 ke 2012 (1.243.103 → 2.049.576), menunjukkan pertumbuhan bisnis yang kuat. Dekomposisi time series mengonfirmasi adanya tren naik yang konsisten, ditambah pola musiman berulang tiap tahun (puncak di pertengahan tahun, terendah di musim dingin).
- **Kesimpulan pertanyaan 2:** Segmentasi K-Means (k=3) berhasil mengelompokkan hari menjadi 3 segmen berbeda: **Ramai** (suhu hangat, kelembapan sedang, ~5.982 penyewaan/hari), **Sedang** (suhu dingin, angin kencang, ~3.341/hari), dan **Sepi** (kelembapan tertinggi, ~3.113/hari). Kombinasi suhu hangat & kelembapan rendah adalah pendorong utama hari-hari dengan penyewaan tinggi.
- **Kesimpulan pertanyaan 3:** Kondisi cuaca berpengaruh signifikan terhadap jumlah penyewaan. Suhu berkorelasi positif (+0,63), sementara kelembapan dan kecepatan angin berkorelasi negatif. Cuaca hujan/salju menurunkan rata-rata penyewaan hingga ~63% dibanding cuaca cerah.
- **Kesimpulan pertanyaan 4:** Pola penyewaan casual dan registered berbeda signifikan antar tipe hari — registered mendominasi hari kerja (pola commuting), sementara proporsi casual meningkat tajam di akhir pekan (pola rekreasi).

In [ ]:
day_df.to_csv("final_data.csv", index=False)